In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn, cuda
from tqdm import tqdm  # Прогресс-бар
import numpy as np
import os

In [ ]:
device = 'cuda' if cuda.is_available() else 'cpu'

In [ ]:
# Параметры
MAX_LEN = 128
TRAIN_BATCH_SIZE = 16
VALID_BATCH_SIZE = 8
EPOCHS = 10  # Увеличиваем количество эпох
LEARNING_RATE = 2e-5
CHECKPOINT_PATH = "sbert_checkpoint_2.pth"  # Путь к файлу сохранения модели
MODEL_WEIGHTS_PATH = "sbert_epoch8_weights_2.pth"  # Путь для сохранения весов на 8-й эпохе

In [ ]:
df = pd.read_excel("data_for_classes.xlsx")

# Переименуем колонки для удобства
df.columns = ['Question', 'Category']

In [ ]:
# Кодирование целевых меток
label_mapping = {label: idx for idx, label in enumerate(df['Category'].unique())}
df['target'] = df['Category'].map(label_mapping)

In [ ]:
label_mapping

{'МОДЕРАЦИЯ': 0,
 'МОНЕТИЗАЦИЯ': 1,
 'УПРАВЛЕНИЕ АККАУНТОМ': 2,
 'ДОСТУП К RUTUBE': 3,
 'ОТСУТСТВУЕТ': 4,
 'ПРЕДЛОЖЕНИЯ': 5,
 'ВИДЕО': 6,
 'ТРАНСЛЯЦИЯ': 7,
 'СОТРУДНИЧЕСТВО ПРОДВИЖЕНИЕ РЕКЛАМА': 8,
 'ПОИСК': 9,
 'БЛАГОТВОРИТЕЛЬНОСТЬ ДОНАТЫ': 10}

In [ ]:
train_data, val_data = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("ai-forever/sbert_large_nlu_ru")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
class QuestionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.texts = dataframe['Question'].tolist()
        self.targets = dataframe['target'].tolist()
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text = str(self.texts[index])
        target = self.targets[index]

        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=True,
            truncation=True
        )

        return {
            'ids': torch.tensor(inputs['input_ids'], dtype=torch.long),
            'mask': torch.tensor(inputs['attention_mask'], dtype=torch.long),
            'targets': torch.tensor(target, dtype=torch.long)
        }

In [ ]:
train_dataset = QuestionDataset(train_data, tokenizer, MAX_LEN)
val_dataset = QuestionDataset(val_data, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=VALID_BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
class SbertClassifier(nn.Module):
    def __init__(self, num_classes):
        super(SbertClassifier, self).__init__()
        # Используем модель sBERT
        self.bert = AutoModel.from_pretrained("ai-forever/sbert_large_nlu_ru")

        # Многослойный классификатор (MLP) с Dropout
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),  # Первый линейный слой с 512 нейронами (1024 - размер скрытого слоя для данной модели)
            nn.ReLU(),            # Функция активации ReLU
            nn.Dropout(0.3),      # Dropout для регуляризации
            nn.Linear(512, 256),  # Второй линейный слой с 256 нейронами
            nn.ReLU(),            # Еще одна ReLU
            nn.Dropout(0.3),      # Дополнительный Dropout
            nn.Linear(256, num_classes)  # Выходной слой с количеством классов
        )

    def forward(self, ids, mask):
        # Извлекаем последний слой скрытых состояний
        _, pooled_output = self.bert(ids, attention_mask=mask, return_dict=False)
        output = self.classifier(pooled_output)  # Прогоняем через многослойный перцептрон
        return output

In [ ]:
# Инициализация модели
num_classes = len(label_mapping)
model = SbertClassifier(num_classes)
#model = model.to(device)

config.json:   0%|          | 0.00/863 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# Функции потерь и оптимизатор
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params=model.parameters(), lr=LEARNING_RATE)

In [ ]:
# Функция сохранения чекпоинта
def save_checkpoint(epoch, model, optimizer, loss, path=CHECKPOINT_PATH):
    state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss
    }
    torch.save(state, path)
    print(f"Checkpoint saved after epoch {epoch+1}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
gdrive_checkpoint_path = "/content/drive/MyDrive/sbert_checkpoint.pth"

# Локальный путь к файлу с весами модели
local_checkpoint_path = "sbert_checkpoint.pth"

# Проверяем, существует ли файл локально
if os.path.isfile(local_checkpoint_path):
    print(f"Загрузка чекпоинта из {local_checkpoint_path}")
    # Загружаем веса модели
    checkpoint = torch.load(local_checkpoint_path)

    # Сохраняем вес на Google Drive
    torch.save(checkpoint, gdrive_checkpoint_path)
    print(f"Чекпоинт сохранен на Google Drive по пути: {gdrive_checkpoint_path}")
else:
    print(f"Локальный файл {local_checkpoint_path} не найден!")

Загрузка чекпоинта из sbert_checkpoint.pth


<ipython-input-27-f4a29e9c208a>:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(local_checkpoint_path)


Чекпоинт сохранен на Google Drive по пути: /content/drive/MyDrive/sbert_checkpoint.pth


In [ ]:
def load_checkpoint(model, optimizer, path=CHECKPOINT_PATH):
    if os.path.isfile(path):
        state = torch.load(path)
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        start_epoch = state['epoch']
        loss = state['loss']
        print(f"Checkpoint loaded, starting from epoch {start_epoch+1}")
        return start_epoch
    else:
        print("No checkpoint found, starting from scratch")
        return 0

In [ ]:
def train_model(epoch):
    model.train()
    total_loss = 0
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Training Epoch {epoch+1}")
    for _, batch in progress_bar:
        ids = batch['ids'].to(device, dtype=torch.long)
        mask = batch['mask'].to(device, dtype=torch.long)
        targets = batch['targets'].to(device, dtype=torch.long)

        optimizer.zero_grad()
        outputs = model(ids, mask)
        loss = criterion(outputs, targets)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

        progress_bar.set_postfix({'Training Loss': f"{total_loss/len(train_loader):.4f}"})
    print(f"Epoch {epoch+1}, Training loss: {total_loss/len(train_loader):.4f}")
    return total_loss/len(train_loader)

In [ ]:
def evaluate_model():
    model.eval()
    fin_targets = []
    fin_outputs = []
    progress_bar = tqdm(enumerate(val_loader), total=len(val_loader), desc="Validating")
    with torch.no_grad():
        for _, batch in progress_bar:
            ids = batch['ids'].to(device, dtype=torch.long)
            mask = batch['mask'].to(device, dtype=torch.long)
            targets = batch['targets'].to(device, dtype=torch.long)

            outputs = model(ids, mask)
            _, preds = torch.max(outputs, dim=1)

            fin_targets.extend(targets.cpu().detach().numpy().tolist())
            fin_outputs.extend(preds.cpu().detach().numpy().tolist())

    return fin_outputs, fin_targets

In [ ]:
start_epoch = load_checkpoint(model, optimizer)

No checkpoint found, starting from scratch


In [ ]:
for epoch in range(start_epoch, EPOCHS):
    train_loss = train_model(epoch)
    outputs, targets = evaluate_model()
    accuracy = np.mean(np.array(outputs) == np.array(targets))
    print(f"Validation Accuracy after Epoch {epoch+1}: {accuracy:.4f}")

    # Сохранение модели после каждой эпохи
    save_checkpoint(epoch, model, optimizer, train_loss)

Training Epoch 1: 100%|██████████| 97/97 [01:43<00:00,  1.06s/it, Training Loss=2.1011]


Epoch 1, Training loss: 2.1011


Validating: 100%|██████████| 49/49 [00:09<00:00,  5.22it/s]


Validation Accuracy after Epoch 1: 0.6269
Checkpoint saved after epoch 1


Training Epoch 2: 100%|██████████| 97/97 [01:44<00:00,  1.08s/it, Training Loss=1.2562]


...

Validation Accuracy after Epoch 9: 0.8782
Checkpoint saved after epoch 9


Training Epoch 10: 100%|██████████| 97/97 [01:44<00:00,  1.08s/it, Training Loss=0.1235]


Epoch 10, Training loss: 0.1235


Validating: 100%|██████████| 49/49 [00:09<00:00,  5.31it/s]


Validation Accuracy after Epoch 10: 0.9016
Checkpoint saved after epoch 10


In [ ]:
start_epoch = load_checkpoint(model, optimizer)

<ipython-input-15-ef0e6a5ea28e>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(path)


Checkpoint loaded, starting from epoch 10


In [ ]:
# Функция предсказания для одного текста
def predict(text, model, tokenizer, max_len=MAX_LEN):
    # Токенизация входного текста
    inputs = tokenizer.encode_plus(
        text,
        None,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        return_token_type_ids=True,
        truncation=True
    )

    # Преобразование токенов в тензоры
    ids = torch.tensor([inputs['input_ids']], dtype=torch.long).to(device)
    mask = torch.tensor([inputs['attention_mask']], dtype=torch.long).to(device)

    # Применение модели для предсказания
    with torch.no_grad():
        outputs = model(ids, mask)

    # Получение предсказанного класса
    _, preds = torch.max(outputs, dim=1)

    # Получение категории из предсказанного индекса
    predicted_label = list(label_mapping.keys())[list(label_mapping.values()).index(preds.item())]

    return predicted_label

# Примеры для тестирования модели
test_texts = [
    "Почему вы не разглашаете, чьи авторские права я нарушаю?",
    "Как выйти из аккаунта в приложении Студия RUTUBE?",
    "Какие требования к обложке видео?",
    "Где находятся настройки доступа к студии канала в Студии RUTUBE?",
    "Есть ли ограничения на одновременное проведение трансляций на RUTUBE и на других площадках (YouTube, Twitch и VK)?"
]

# Проверка инференса на нескольких примерах
print("Инференс для тестовых примеров:")
for text in test_texts:
    result = predict(text, model, tokenizer)
    print(f"Текст: {text} => Предсказанная категория: {result}")

Инференс для тестовых примеров:
Текст: Почему вы не разглашаете, чьи авторские права я нарушаю? => Предсказанная категория: МОДЕРАЦИЯ
Текст: Как выйти из аккаунта в приложении Студия RUTUBE? => Предсказанная категория: УПРАВЛЕНИЕ АККАУНТОМ
Текст: Какие требования к обложке видео? => Предсказанная категория: ВИДЕО
Текст: Где находятся настройки доступа к студии канала в Студии RUTUBE? => Предсказанная категория: УПРАВЛЕНИЕ АККАУНТОМ
Текст: Есть ли ограничения на одновременное проведение трансляций на RUTUBE и на других площадках (YouTube, Twitch и VK)? => Предсказанная категория: ТРАНСЛЯЦИЯ
